# Optional Lab 8C — Real A2A

Chapter 8 called its envelope "A2A". This lab makes that literal.

**A2A** — Google's Agent2Agent protocol — is to agents what MCP is to tools: an open
wire protocol for one agent to discover another and hand it work. Chapter 3 used the
real MCP SDK; this lab uses the real `a2a-sdk` (protocol 1.0), so the book has both
protocols the industry currently pairs.

The rule is the same as Lab 8B: the *team* is imported from Chapter 8 and does not
change. The protocol is allowed to decide exactly two things:

| Chapter 8 (in-process) | A2A protocol (on the wire) |
|---|---|
| the orchestrator imports the workers | the orchestrator **discovers** them by **Agent Card** |
| `A2AMessage` passed as a Python object | the envelope travels as a data part of a **Message** |
| a function call returns | a **Task** moves `submitted → working → completed` and returns an **Artifact** |
| `trace_id` on the envelope | `context_id` on every task — the protocol's own name for it |

Same verdict, same handoffs, same audit, one trace — or the claim is false.


## Setup
[link text](https://)
This lab installs from **one** `requirements.txt`file.


In [8]:
REPO_URL = "https://github.com/gstripling00/ai-engineer.git"

import os, sys, subprocess

if not os.path.isdir("aegis"):
    result = subprocess.run(["git", "clone", REPO_URL, "aegis"],
                            capture_output=True, text=True)
    if result.returncode != 0:
        raise RuntimeError("git clone failed - check REPO_URL above.\n" + result.stderr)

os.chdir("aegis")
sys.path.insert(0, os.path.abspath("."))
print("repo:", os.getcwd())


repo: /content/aegis/aegis


In [2]:
# --no-warn-conflicts silences a cosmetic Colab-only notice about `requests`;
# see the comment block at the top of requirements.txt. Real resolver errors still raise.
!pip -q install --no-warn-conflicts -r requirements.txt


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 247.5/247.5 kB 12.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 558.3/558.3 kB 24.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 83.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.1/4.1 MB 108.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 222.6/222.6 kB 21.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 466.5/466.5 kB 40.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.8/71.8 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 246.0/246.0 kB 23.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.6/121.6 kB 12.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 11.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Verify the environment and confirm this lab's source folder is in the checkout.


In [3]:
!python tools/check_env.py --chapter 8c


dependencies
  ok      langgraph              open-source agent track (StateGraph/END)
  ok      langchain-core         message and tool primitives
  ok      langchain-community    RAGAS dependency — see the pin note
  ok      google-adk             Google Cloud agent track (Agent, Workflow)
  ok      mcp                    tool discovery and hardening (Ch 3, 9, 11)
  ok      a2a-sdk                agent-to-agent protocol (Lab 8C)
  ok      openai                 the default real-model tier
  ok      langchain-openai       wires OpenAI into RAGAS
  ok      ragas                  evaluation (Ch 10)
  ok      sacrebleu              required by RAGAS BleuScore
  ok      opentelemetry-sdk      tracing (Ch 10)
  ok      chromadb               vector store (Ch 6)
  ok      rank-bm25              sparse retrieval for hybrid search (Ch 6)
  ok      pytest                 the test suite

critical pin
  ok      langchain-community 0.3.29 (compatible with ragas)

model access
  absent  OPENAI_API

In [4]:
import os

os.environ["AEGIS_MODEL"] = "mock"     # free, deterministic, no key
print("model tier:", os.environ["AEGIS_MODEL"])


model tier: mock


## Three agents, three cards

Each Chapter 8 worker becomes an A2A server: a FastAPI app with the card at
`/.well-known/agent-card.json` and a JSON-RPC endpoint at `/`. The card says what the
agent is, what skill it offers, and — in its tags — which tools it holds. That last
part is the least-privilege story made *discoverable*: a client can see that only
`reporting` advertises `create_ticket` before it sends anything.

The three apps are mounted on an in-process transport, so this runs offline. Replace
the transport with real URLs and nothing else changes.


In [5]:
import sys, json
sys.path.insert(0, "labs/chapter-08c-real-a2a")

from a2a_team.agents import build_agent_app, make_card, SPECS
from a2a_team.orchestrator import AGENT_URLS, MultiAppTransport

apps = {url.split("//")[1]: build_agent_app(name, url + "/") for name, url in AGENT_URLS.items()}
for host, app in apps.items():
    print(f"{host:14} routes: {[r.path for r in app.routes if 'well-known' in r.path or r.path == '/']}")

print()
card = make_card("reporting", "http://reporting/")
print("the reporting agent's card, as a client would read it:")
print("  name:       ", card.name)
print("  description:", card.description)
print("  interface:  ", card.supported_interfaces[0].url, card.supported_interfaces[0].protocol_binding)
print("  skill:      ", card.skills[0].name)
print("  tags:       ", list(card.skills[0].tags))


triage         routes: ['/.well-known/agent-card.json', '/']
investigation  routes: ['/.well-known/agent-card.json', '/']
reporting      routes: ['/.well-known/agent-card.json', '/']

the reporting agent's card, as a client would read it:
  name:        reporting
  description: Opens the ticket. The ONLY agent that writes to the world.
  interface:   http://reporting/ JSONRPC
  skill:       reporting an Aegis alert
  tags:        ['soc', 'aegis', 'tool:create_ticket']


## Discovery, then a task

The orchestrator knows three URLs and nothing else. It fetches each card, then opens
Chapter 8's envelope and sends it to whichever agent the envelope's `to_agent` names —
as a `Message` whose `context_id` is the incident's trace id. The agent runs the
unchanged worker inside a `Task`, attaches the next envelope as an `Artifact`, and
completes. The orchestrator reads the artifact, sees the new `to_agent`, and repeats.


In [6]:
import httpx
from a2a_team.agents import discover, send_envelope, workers
from a2a_team.orchestrator import run_incident
from common import soc
from common.a2a import new_investigation

async def one_incident():
    soc.reset_tickets()
    async with httpx.AsyncClient(transport=MultiAppTransport(apps)) as http:
        cards = {name: await discover(http, url) for name, url in AGENT_URLS.items()}
        print("discovered:", {n: [s.name for s in c.skills][0] for n, c in cards.items()})
        print()
        envelope = new_investigation(soc.SEED_ALERT, trace_id="inc-8c")
        while envelope.to_agent in cards:
            result = await send_envelope(http, cards[envelope.to_agent], envelope)
            nxt = result["envelope"]
            print(f'task {result["task_id"][:8]}  context={result["context_id"]}  '
                  f'{result["state"].replace("TASK_STATE_", ""):9}  {envelope.to_agent:14} -> {nxt.to_agent}')
            envelope = nxt
        return envelope

final = await one_incident()          # notebook kernels have a running loop: top-level await
print()
print("verdict:", final.payload["verdict"], "| severity:", final.payload["severity"], "| ticket:", final.payload["ticket"]["id"])


discovered: {'triage': 'triage an Aegis alert', 'investigation': 'investigation an Aegis alert', 'reporting': 'reporting an Aegis alert'}

task 2a3add62  context=inc-8c  COMPLETED  triage         -> investigation
task 70405356  context=inc-8c  COMPLETED  investigation  -> reporting
task 6540706d  context=inc-8c  COMPLETED  reporting      -> orchestrator

verdict: confirmed_compromise | severity: critical | ticket: INC-1001


Three tasks, three different task ids, one `context_id`. That is the protocol's own
version of Chapter 8's single trace id — and it is the field a Langfuse or OTel trace
would join on if these agents ran on three different machines.

## The comparison with Chapter 8

`run_in_process()` does the whole thing in one call and also captures the audit trail,
so the protocol run can be checked field by field against Chapter 8's in-process run.


In [7]:
from a2a_team.orchestrator import run_in_process

soc.reset_tickets()
a2a_run = run_in_process(soc.SEED_ALERT, trace_id="inc-8c")

# Chapter 8, in-process, same alert
soc.reset_tickets(); mark = len(workers.AUDIT)
from common.model import get_model
m = new_investigation(soc.SEED_ALERT, trace_id="inc-8c")
hops = []
for stage in (workers.triage, workers.investigate, workers.report):
    before = m.to_agent; m = stage(m, get_model()); hops.append(f"{before} -> {m.to_agent}")
ch8_audit = [(e["role"], e["tool"], e["allowed"]) for e in workers.AUDIT[mark:]]

print("same verdict:     ", a2a_run["final"].payload["verdict"] == m.payload["verdict"], "->", m.payload["verdict"])
print("same handoffs:    ", a2a_run["hops"] == hops, "->", hops)
print("same audit trail: ", a2a_run["audit"] == ch8_audit, f"({len(ch8_audit)} authorization decisions)")
print("one context id:   ", {t["context_id"] for t in a2a_run["tasks"]})
print("who touched the world:", {a for a, t, ok in a2a_run["audit"] if t == "create_ticket" and ok})


same verdict:      True -> confirmed_compromise
same handoffs:     True -> ['triage -> investigation', 'investigation -> reporting', 'reporting -> orchestrator']
same audit trail:  True (5 authorization decisions)
one context id:    {'inc-8c'}
who touched the world: {'reporting'}


## What the protocol changed, and what it did not

It did not change the answer. It changed three things that matter when the agents
are not in the same process, or the same team, or the same company:

- **Discovery is a document, not an import.** The orchestrator never imported a
  worker. A new agent is a new card at a URL — the same move as MCP tool discovery in
  Chapter 9, one level up.
- **A task has a lifecycle.** `submitted → working → completed` (or `failed`, with the
  reason on the task) is visible to the caller. Chapter 8's function call either
  returned or raised; a protocol task can be polled, resumed, and audited.
- **The trace id has a name the whole ecosystem agrees on.** `context_id` is not
  Aegis's convention; it is the protocol's.

And one thing to notice in the card: the tags advertise which tools each agent holds.
That is Chapter 11's supply-chain warning arriving from the other direction — a card
is prose from a server you may not own, and Chapter 11's screening applies to it
exactly as it did to MCP tool descriptions.

---

## What you built

Chapter 8's team as three real A2A agents: cards, discovery, tasks, artifacts, one
`context_id` — producing the same verdict, handoffs, and audit trail as the in-process
version.

- **Two protocols, one pattern.** MCP describes tools; A2A describes agents. Both are
  discovery documents plus a call contract, and both are untrusted text.
- **The envelope survived unchanged**, which is why Chapter 8 built one.
- **In-process today, three machines tomorrow.** Only the transport changes.
